In [1]:
def save_frames_as_images(ds, save_folder, img_size, meta_path):
    # 동아시아 영역 자르기 (위도 방향 주의)
    ds_eastasia = ds.sel(latitude=slice(43, 33), longitude=slice(124, 134))

    # lev 차원 제거 (있으면)
    if 'lev' in ds_eastasia['t2m'].dims:
        data = ds_eastasia['t2m'].squeeze('lev').values
    else:
        data = ds_eastasia['t2m'].values
    
    # 결측치 -> 0
    # data = np.nan_to_num(data, nan=0.0)
    nan_mask = np.isnan(data)

    # 최소/최대값
    min_val = float(np.nanmin(data))
    max_val = float(np.nanmax(data))

    # 0~255 정규화
    normed_data = (data - min_val) / (max_val - min_val) * 255
    normed_data = normed_data.astype(np.uint8)

    normed_data[nan_mask]=0
    normed_data = normed_data.astype(np.uint8)

    # 폴더 생성
    os.makedirs(save_folder, exist_ok=True)

    # 프레임 이미지 저장
    for idx, frame in enumerate(tqdm(normed_data)):
        img = Image.fromarray(frame)  # 위아래 반전
        img = img.resize(img_size, resample=Image.BILINEAR)
        img = img.convert('L')
        img.save(os.path.join(save_folder, f"{idx:05d}.png"))

    # 메타데이터 저장
    metadata = {
        "min_val": min_val,
        "max_val": max_val,
        'uint': "K",
        "lat_range": [float(ds_eastasia.latitude.max()), float(ds_eastasia.latitude.min())],
        "lon_range": [float(ds_eastasia.longitude.min()), float(ds_eastasia.longitude.max())],
        'original_shape': list(data.shape),
        'time': [str(t) for t in ds_eastasia.valid_time.values]
    }

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)


In [2]:
# ds_lr = xr.open_dataset(LR_PATH)
# ds_hr = xr.open_dataset(HR_PATH)
import os
import json
import numpy as np
import xarray as xr
from tqdm import tqdm
from PIL import Image

era_land_path = "../../../data/ERA/era_land_t2m_merged.nc"
era_path = "../../../data/ERA/era_t2m_merged.nc"
ds_era = xr.open_dataset(era_path)
ds_era_land = xr.open_dataset(era_land_path)

SAVE_DIR = "./load/climate"
# os.makedirs(os.path.join(SAVE_DIR, 'climate_train_korea_imerg50'), exist_ok=True)

# === Train ===
train_time_slice = slice('2000-01-01', '2017-12-31')
# save_frames_as_images(0
#     ds.sel(time=train_time_slice),
#     save_folder=os.path.join(SAVE_DIR, 'climate_train_korea_imerg50'),
#     img_size=(50, 50),
#     meta_path=os.path.join(SAVE_DIR, 'meta', 'train_imerg50_meta.json')
# )

# # era_land
save_frames_as_images(
    ds_era_land.sel(valid_time=train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_t2m_eraLand'),
    img_size=(100, 100),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'train_eraLand_t2m.json')
)

# # era
save_frames_as_images(
    ds_era.sel(valid_time=train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_t2m_era'),
    img_size=(41, 41),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'train_era_t2m.json')
)


# === Validation ===
val_time_slice = slice('2018-01-01', '2020-12-31')
# save_frames_as_images(
#     ds.sel(time=val_time_slice),
#     save_folder=os.path.join(SAVE_DIR, 'climate_val_korea_imerg50'),
#     img_size=(50, 50),
#     meta_path=os.path.join(SAVE_DIR, 'meta', 'val_imerg50_meta.json')
# )

# era_land
save_frames_as_images(
    ds_era_land.sel(valid_time=val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_t2m_eraLand'),
    img_size=(100, 100),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'val_eraLand_t2m.json')
)

# # era
save_frames_as_images(
    ds_era.sel(valid_time=val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_t2m_era'),
    img_size=(41, 41),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'val_era_t2m.json')
)


/tmp/ipykernel_2537109/2094157113.py:21: RuntimeWarning: invalid value encountered in cast
  normed_data = normed_data.astype(np.uint8)
100%|██████████| 6575/6575 [00:01<00:00, 5331.34it/s]
/tmp/ipykernel_2537109/2094157113.py:21: RuntimeWarning: invalid value encountered in cast
  normed_data = normed_data.astype(np.uint8)
100%|██████████| 1096/1096 [00:00<00:00, 5618.78it/s]


In [4]:
ds = xr.open_dataset(era_land_path)
display(ds.t2m.isel(valid_time=0).values)

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)